In [3]:
%pip install pandas numpy matplotlib seaborn
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
df = pd.read_csv("C:\\internship\\datasets\\processed\\diabetic_data_base_table_mapped.csv", low_memory=False)

print(df.shape)
df.head()


(101766, 51)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmit_30
0,2278392,8222157,Caucasian,Female,[0-10),NaN,NaN,Not Mapped,Physician Referral,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO,0
1,149190,55629189,Caucasian,Female,[10-20),NaN,Emergency,Discharged to home,Emergency Room,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30,0
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,Emergency,Discharged to home,Emergency Room,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO,0
3,500364,82442376,Caucasian,Male,[30-40),NaN,Emergency,Discharged to home,Emergency Room,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO,0
4,16680,42519267,Caucasian,Male,[40-50),NaN,Emergency,Discharged to home,Emergency Room,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,0


In [5]:
def map_diagnosis(code):
    
    try:
        code = float(code)
    except:
        return "Other"
    
    if 390 <= code < 460:
        return "Circulatory"
    elif 460 <= code < 520:
        return "Respiratory"
    elif 520 <= code < 580:
        return "Digestive"
    elif 250 <= code < 251:
        return "Diabetes"
    elif 800 <= code < 1000:
        return "Injury"
    else:
        return "Other"

In [6]:
df["primary_diagnosis_group"] = df["diag_1"].apply(map_diagnosis)

In [7]:

freq = df["primary_diagnosis_group"].value_counts(normalize=True)

rare_groups = freq[freq < 0.02].index

In [8]:
df["primary_diagnosis_group_reduced"] = df["primary_diagnosis_group"].replace(
    rare_groups,
    "Other"
)

In [9]:
# =========================
# DIAGNOSIS FEATURES
# =========================

def map_diag_category(diag):
    try:
        diag = float(diag)
        if 390 <= diag < 460:
            return 'circulatory'
        elif 460 <= diag < 520:
            return 'respiratory'
        elif 520 <= diag < 580:
            return 'digestive'
        elif 250 <= diag < 251:
            return 'diabetes'
        else:
            return 'other'
    except:
        return 'other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col + '_group'] = df[col].apply(map_diag_category)

df['has_circulatory'] = (
    (df['diag_1_group'] == 'circulatory') |
    (df['diag_2_group'] == 'circulatory') |
    (df['diag_3_group'] == 'circulatory')
).astype(int)

df['has_respiratory'] = (
    (df['diag_1_group'] == 'respiratory') |
    (df['diag_2_group'] == 'respiratory') |
    (df['diag_3_group'] == 'respiratory')
).astype(int)

df['has_diabetes_diag'] = (
    (df['diag_1_group'] == 'diabetes') |
    (df['diag_2_group'] == 'diabetes') |
    (df['diag_3_group'] == 'diabetes')
).astype(int)

df['num_unique_diag_groups'] = df[
    ['diag_1_group', 'diag_2_group', 'diag_3_group']
].nunique(axis=1)

df['num_non_other_diag'] = (
    (df[['diag_1_group','diag_2_group','diag_3_group']] != 'other')
).sum(axis=1)

In [10]:
age_order = sorted(df["age"].unique())

age_map = {age:i for i,age in enumerate(age_order)}

df["age_ordinal"] = df["age"].map(age_map)

In [11]:
low_risk = ["[0-10)","[10-20)","[20-30)","[30-40)","[40-50)"]
medium_risk = ["[50-60)","[60-70)"]
high_risk = ["[70-80)","[80-90)","[90-100)"]

In [12]:
def age_bucket(x):

    if x in low_risk:
        return "low"
    elif x in medium_risk:
        return "medium"
    else:
        return "high"

In [13]:
df["age_risk_group"] = df["age"].apply(age_bucket)

In [14]:
df["prior_inpatient_flag"] = (df["number_inpatient"] >= 1).astype(int)

In [15]:
medication_cols = [
'metformin','repaglinide','nateglinide','chlorpropamide',
'glimepiride','acetohexamide','glipizide','glyburide',
'tolbutamide','pioglitazone','rosiglitazone','acarbose',
'miglitol','troglitazone','tolazamide','examide',
'citoglipton','insulin','glyburide-metformin',
'glipizide-metformin','glimepiride-pioglitazone',
'metformin-rosiglitazone','metformin-pioglitazone'
]

In [16]:
# =========================
# MEDICATION FEATURES
# =========================

med_cols = [
    'metformin','repaglinide','nateglinide','chlorpropamide',
    'glimepiride','acetohexamide','glipizide','glyburide',
    'tolbutamide','pioglitazone','rosiglitazone','acarbose',
    'miglitol','troglitazone','tolazamide','examide',
    'citoglipton','insulin','glyburide-metformin',
    'glipizide-metformin','glimepiride-pioglitazone',
    'metformin-rosiglitazone','metformin-pioglitazone'
]

df['num_active_medications'] = (df[med_cols] != 'No').sum(axis=1)

df['insulin_active'] = (df['insulin'] != 'No').astype(int)

df['med_change_intensity'] = (
    (df[med_cols] == 'Up') | (df[med_cols] == 'Down')
).sum(axis=1)

df['med_stable'] = (df['med_change_intensity'] == 0).astype(int)

In [17]:
df["medication_burden"] = (df[medication_cols] != "No").sum(axis=1)

In [18]:
df["medication_burden_bucket"] = df["medication_burden"].clip(upper=3)

In [19]:
df["on_insulin"] = (df["insulin"] != "No").astype(int)

In [20]:
df["med_change_flag"] = (df["change"] == "Ch").astype(int)

In [21]:
df["diabetes_med_flag"] = (df["diabetesMed"] == "Yes").astype(int)

In [22]:
# =========================
# UTILIZATION FEATURES
# =========================

df['total_visits'] = (
    df['number_outpatient'] +
    df['number_emergency'] +
    df['number_inpatient']
)

df['total_visits'] = df['total_visits'].replace(0, 1)

df['emergency_ratio'] = df['number_emergency'] / df['total_visits']
df['inpatient_ratio'] = df['number_inpatient'] / df['total_visits']

df['visit_intensity'] = df['total_visits'] / df['time_in_hospital']
df['high_utilization'] = (df['number_inpatient'] >= 2).astype(int)

In [23]:
# =========================
# CLINICAL COMPLEXITY FEATURES
# =========================

df['procedure_density'] = df['num_procedures'] / df['time_in_hospital']
df['diagnosis_complexity'] = df['number_diagnoses']

In [24]:
# =========================
# LAB FEATURES
# =========================

df['max_glu_serum'] = df['max_glu_serum'].replace({'None': 'No'})
df['A1Cresult'] = df['A1Cresult'].replace({'None': 'No'})

# Binary flags
df['high_glucose_flag'] = df['max_glu_serum'].isin(['>200', '>300']).astype(int)
df['high_A1C_flag'] = df['A1Cresult'].isin(['>7', '>8']).astype(int)

# Ordinal encoding
glu_map = {'No':0, 'Norm':1, '>200':2, '>300':3}
a1c_map = {'No':0, 'Norm':1, '>7':2, '>8':3}

df['glu_level'] = df['max_glu_serum'].map(glu_map)
df['a1c_level'] = df['A1Cresult'].map(a1c_map)

In [25]:
# =========================
# INTERACTION FEATURES
# =========================

df['meds_x_time'] = df['num_medications'] * df['time_in_hospital']
df['inpatient_x_meds'] = df['number_inpatient'] * df['num_medications']
df['labs_x_time'] = df['num_lab_procedures'] * df['time_in_hospital']
df['age_x_meds'] = df['age_ordinal'] * df['num_medications']
df['inpatient_x_time'] = df['number_inpatient'] * df['time_in_hospital']

In [26]:
model_features = [

"readmit_30",

"primary_diagnosis_group_reduced",

"age_ordinal",
"age_risk_group",

"time_in_hospital",

"num_lab_procedures",
"num_medications",

"number_emergency",
"number_outpatient",
"number_inpatient",

"prior_inpatient_flag",

"on_insulin",
"med_change_flag",
"diabetes_med_flag",

"medication_burden_bucket"

]
model_features += [
    'total_visits','emergency_ratio','inpatient_ratio','visit_intensity','high_utilization',
    'procedure_density','diagnosis_complexity',
    'high_glucose_flag','high_A1C_flag','glu_level','a1c_level',
    'has_circulatory','has_respiratory','has_diabetes_diag',
    'num_unique_diag_groups','num_non_other_diag',
    'num_active_medications','insulin_active','med_change_intensity','med_stable',
    'meds_x_time','inpatient_x_meds','labs_x_time','age_x_meds','inpatient_x_time'
]

In [27]:
model_df = df[model_features].copy()

In [28]:
model_df.fillna(0, inplace=True)

In [29]:
model_df.isna().sum()

readmit_30                         0
primary_diagnosis_group_reduced    0
age_ordinal                        0
age_risk_group                     0
time_in_hospital                   0
num_lab_procedures                 0
num_medications                    0
number_emergency                   0
number_outpatient                  0
number_inpatient                   0
prior_inpatient_flag               0
on_insulin                         0
med_change_flag                    0
diabetes_med_flag                  0
medication_burden_bucket           0
total_visits                       0
emergency_ratio                    0
inpatient_ratio                    0
visit_intensity                    0
high_utilization                   0
procedure_density                  0
diagnosis_complexity               0
high_glucose_flag                  0
high_A1C_flag                      0
glu_level                          0
a1c_level                          0
has_circulatory                    0
h

In [30]:
assert "readmit_30" in model_df.columns

In [31]:
model_df.head()
model_df.shape

(101766, 40)

In [32]:
import os


os.makedirs("C:\\internship\\datasets\\processed", exist_ok=True)

model_df.to_csv(
    "C:\\internship\\datasets\\processed\\diabetes_model_base.csv",
    index=False
)

In [33]:
feature_dictionary = pd.DataFrame({

"Feature": model_df.columns,

"Source":[
"Target" if col=="readmit_30" else "EDA Phase 2"
for col in model_df.columns
],

"Description":[
"30-day readmission outcome"
if col=="readmit_30"
else "Engineered feature"
for col in model_df.columns
]

})

feature_dictionary

,Feature,Source,Description
0,readmit_30,Target,30-day readmission outcome
1,primary_diagnosis_group_reduced,EDA Phase 2,Engineered feature
2,age_ordinal,EDA Phase 2,Engineered feature
3,age_risk_group,EDA Phase 2,Engineered feature
4,time_in_hospital,EDA Phase 2,Engineered feature
5,num_lab_procedures,EDA Phase 2,Engineered feature
6,num_medications,EDA Phase 2,Engineered feature
7,number_emergency,EDA Phase 2,Engineered feature
8,number_outpatient,EDA Phase 2,Engineered feature
9,number_inpatient,EDA Phase 2,Engineered feature
